In [1]:
# installing libraries 
!pip install langchain langchain-groq youtube-transcript-api yt-dlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 11.6 MB/s  0:00:00 eta 0:00:01


In [3]:
!pip show yt-dlp

Name: yt-dlp
Version: 2026.6.9
Summary: A feature-rich command-line audio/video downloader
Home-page: 
Author: 
Author-email: pukkandan <pukkandan.ytdlp@gmail.com>
License-Expression: Unlicense
Location: /opt/anaconda3/lib/python3.11/site-packages
Requires: 
Required-by: 


In [4]:
import sys
print(sys.executable)
print(sys.version)

/Users/disha/opt/anaconda3/bin/python
3.9.12 (main, Apr  5 2022, 01:53:17) 
[Clang 12.0.0 ]


In [5]:
import sys
!{sys.executable} -m pip install yt-dlp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 12.3 MB/s  0:00:00 eta 0:00:01


In [6]:
# importing libraries
import os
import re
import json

from langchain_groq import ChatGroq
from langchain.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

from youtube_transcript_api import YouTubeTranscriptApi
import yt_dlp

In [10]:
# set up LLM 
os.environ["GROQ_API_KEY"] = "gsk_
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

In [11]:
# testing LLM
response = llm.invoke("Say hello in one sentence.")
print(response.content)

Hello, how can I assist you today?


In [12]:
# vid ID extraction tool - TOOL 1 
@tool
def extract_video_id(url: str) -> str:
    """
    Extracts a YouTube video ID from a YouTube URL.
    """

    patterns = [
        r"v=([a-zA-Z0-9_-]{11})",
        r"youtu\.be/([a-zA-Z0-9_-]{11})"
    ]

    for pattern in patterns:
        match = re.search(pattern, url)
        if match:
            return match.group(1)

    return "Video ID not found."

In [13]:
# testing tool
extract_video_id.invoke(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

'dQw4w9WgXcQ'

In [19]:
# transcript fetching tool - TOOL 2
@tool
def get_transcript(video_id: str) -> str:
    """
    Fetches the transcript of a YouTube video.

    Input:
    video_id: YouTube video ID

    Output:
    Transcript text, shortened to 3000 characters.
    """
    try:
        ytt_api = YouTubeTranscriptApi()
        transcript = ytt_api.fetch(video_id)

        text = " ".join([entry.text for entry in transcript])

        return text[:3000]

    except Exception as e:
        return f"Error fetching transcript: {str(e)}"

In [20]:
# testing tool 
get_transcript.invoke("dQw4w9WgXcQ")

"[♪♪♪] ♪ We're no strangers to love ♪ ♪ You know the rules\nand so do I ♪ ♪ A full commitment's\nwhat I'm thinking of ♪ ♪ You wouldn't get this\nfrom any other guy ♪ ♪ I just wanna tell you\nhow I'm feeling ♪ ♪ Gotta make you understand ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around\nand desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie\nand hurt you ♪ ♪ We've known each other\nfor so long ♪ ♪ Your heart's been aching\nbut you're too shy to say it ♪ ♪ Inside we both know\nwhat's been going ♪ ♪ We know the game\nand we're gonna play it ♪ ♪ And if you ask me\nhow I'm feeling ♪ ♪ Don't tell me\nyou're too blind to see ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around\nand desert you ♪ ♪ Never gonna make you cry ♪ ♪ Never gonna say goodbye ♪ ♪ Never gonna tell a lie\nand hurt you ♪ ♪ Never gonna give you up ♪ ♪ Never gonna let you down ♪ ♪ Never gonna run around\nand desert y

In [22]:
# youtube metadata tool - TOOL 3 
@tool
def get_video_metadata(video_url: str) -> str:
    """
    Retrieves metadata from a YouTube video.
    """

    try:
        ydl_opts = {
            "quiet": True,
            "skip_download": True
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)

        metadata = {
            "title": info.get("title"),
            "channel": info.get("uploader"),
            "duration": info.get("duration"),
            "views": info.get("view_count")
        }

        return json.dumps(metadata, indent=2)

    except Exception as e:
        return f"Error: {str(e)}"

In [23]:
# testing tool 
get_video_metadata.invoke(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


'{\n  "title": "Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)",\n  "channel": "Rick Astley",\n  "duration": 213,\n  "views": 1785621044\n}'

In [24]:
# thumbnail retrieval - TOOL 4
@tool
def get_thumbnail(video_url: str) -> str:
    """
    Retrieves the thumbnail URL for a YouTube video.
    """

    try:
        ydl_opts = {
            "quiet": True,
            "skip_download": True
        }

        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            info = ydl.extract_info(video_url, download=False)

        return info.get("thumbnail", "Thumbnail not found")

    except Exception as e:
        return f"Error: {str(e)}"

In [25]:
# testing tool 
get_thumbnail.invoke(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


'https://i.ytimg.com/vi_webp/dQw4w9WgXcQ/maxresdefault.webp'

In [26]:
# tool list 
tools = [extract_video_id, get_transcript, get_video_metadata, get_thumbnail]

In [27]:
# tool map
tool_map = {tool.name: tool for tool in tools}

In [28]:
# testing
tool_map.keys()

dict_keys(['extract_video_id', 'get_transcript', 'get_video_metadata', 'get_thumbnail'])

In [29]:
# binding tools to LLM 
llm_with_tools = llm.bind_tools(tools)

In [30]:
# testing if LLM will pick a tool 
response = llm_with_tools.invoke(
    "Get the metadata for this video: https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

response.tool_calls

[{'name': 'get_video_metadata',
  'args': {'video_url': 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'},
  'id': 'ywnpj0tq1',
  'type': 'tool_call'}]

In [31]:
# tool call info 
tool_call = response.tool_calls[0]

print(tool_call)

{'name': 'get_video_metadata', 'args': {'video_url': 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'}, 'id': 'ywnpj0tq1', 'type': 'tool_call'}


In [32]:
# extracting pieces
tool_name = tool_call["name"]
tool_args = tool_call["args"]

print("Tool:", tool_name)
print("Args:", tool_args)

Tool: get_video_metadata
Args: {'video_url': 'https://www.youtube.com/watch?v=dQw4w9WgXcQ'}


In [33]:
# execute selected tool 
result = tool_map[tool_name].invoke(tool_args)

print(result)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


{
  "title": "Rick Astley - Never Gonna Give You Up (Official Video) (4K Remaster)",
  "channel": "Rick Astley",
  "duration": 213,
  "views": 1785622647
}


In [34]:
# automated tool calling function
def call_tool_once(user_query: str):
    response = llm_with_tools.invoke(user_query)

    if not response.tool_calls:
        return response.content

    tool_call = response.tool_calls[0]
    tool_name = tool_call["name"]
    tool_args = tool_call["args"]

    result = tool_map[tool_name].invoke(tool_args)

    return result

In [35]:
# testing function
call_tool_once(
    "Get the thumbnail for this video: https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


'https://i.ytimg.com/vi_webp/dQw4w9WgXcQ/maxresdefault.webp'

In [36]:
# asking LLM to explain its result
def call_tool_and_answer(user_query: str):
    messages = [HumanMessage(content=user_query)]

    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        return response.content

    messages.append(response)

    for tool_call in response.tool_calls:
        tool_result = tool_map[tool_call["name"]].invoke(tool_call["args"])

        messages.append(
            ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call["id"]
            )
        )

    final_response = llm_with_tools.invoke(messages)

    return final_response.content

In [37]:
# testing
call_tool_and_answer(
    "Get the metadata for this video: https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


"The function 'get_video_metadata' returns a dictionary with the video's metadata. The metadata includes the title, channel, duration, and views of the video."

In [38]:
# summarisation chain
def summarize_youtube_video(video_url: str):
    video_id = extract_video_id.invoke(video_url)

    transcript = get_transcript.invoke(video_id)

    summary_prompt = f"""
    Summarize this YouTube transcript in 5 bullet points:

    {transcript}
    """

    response = llm.invoke(summary_prompt)

    return response.content

In [39]:
# testing
summarize_youtube_video(
    "https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

'Here are 5 bullet points summarizing the YouTube transcript:\n\n• The speaker is expressing their deep commitment to the person they\'re addressing, stating that they\'re willing to make a full commitment and be honest with them.\n• They\'re acknowledging the emotional connection they share, with the person\'s heart aching due to unspoken feelings, and they\'re urging the person to be open about their emotions.\n• The speaker is reassuring the person that they\'ll never give up on them, let them down, or cause them pain, and that they\'re committed to being honest and true.\n• The song\'s chorus repeats the phrase "never gonna give you up" multiple times, emphasizing the speaker\'s commitment and devotion.\n• The overall message is one of love, commitment, and reassurance, with the speaker urging the person to be open and honest about their feelings and trusting that they\'ll be there to support and care for them.'

In [40]:
# recursive chain
def process_query(messages):
    response = llm_with_tools.invoke(messages)

    if not response.tool_calls:
        return response

    messages.append(response)

    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]

        result = tool_map[tool_name].invoke(tool_args)

        messages.append(
            ToolMessage(
                content=str(result),
                tool_call_id=tool_call["id"]
            )
        )

    return process_query(messages)

In [41]:
# testing
messages = [
    HumanMessage(
        content="Get the metadata for this video: https://www.youtube.com/watch?v=dQw4w9WgXcQ"
    )
]

result = process_query(messages)

print(result.content)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


The function 'get_video_metadata' returns a dictionary with the video's metadata. The metadata includes the title, channel, duration, and views of the video.


In [42]:
# universal chain
def youtube_agent(user_query: str):
    messages = [
        HumanMessage(content=user_query)
    ]

    result = process_query(messages)

    return result.content

In [43]:
# testing chain 
youtube_agent(
    "Get the metadata for https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


"The function 'get_video_metadata' returns a dictionary with the video's metadata. The metadata includes the title, channel, duration, and views of the video."

In [44]:
youtube_agent(
    "Get the thumbnail for https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

Deprecated Feature: Support for Python version 3.9 has been deprecated. Please update to Python 3.10 or above


'The thumbnail URL for the given YouTube video is https://i.ytimg.com/vi_webp/dQw4w9WgXcQ/maxresdefault.webp.'

In [45]:
youtube_agent(
    "Summarize the transcript of https://www.youtube.com/watch?v=dQw4w9WgXcQ"
)

'The song is "Never Gonna Give You Up" by Rick Astley.'